# T1 MRI — QC Filtering
**ABCD Study Release 6.0**

Loads cortical (Desikan-Killiany) and subcortical (ASEG) volumetric parquet files,  
applies the T1 inclusion QC flag, and saves a clean CSV ready for normative modelling.

**Required files** (set paths in the cell below):
```
phenotype/
├── mr_y_smri__vol__dsk.parquet   ← Desikan-Killiany cortical volumes
├── mr_y_smri__vol__aseg.parquet  ← ASEG subcortical volumes
└── mr_y_qc__incl.parquet         ← QC inclusion flags
```


## Configuration

In [1]:
from pathlib import Path

# ── Set these two paths ──────────────────────────────────────
# PHENOTYPE_DIR = Path("path/to/abcd6.0/phenotype")
# OUTPUT_FILE   = Path("output/T1_processed.csv")

PHENOTYPE_DIR = Path("/Users/nioushad/Documents/Doc_p/abcd6.0/phenotype/")
OUTPUT_FILE   = Path("/Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/T1_processed.csv")
# ────────────────────────────────────────────────────────────

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)


## Imports

In [2]:
import pandas as pd

print("pandas", pd.__version__)


pandas 2.2.3


## Step 1 — Load cortical volumes (Desikan-Killiany)

In [3]:
CORTICAL_COLS = [
    # left hemisphere — 34 regions
    'mr_y_smri__vol__dsk__bstmps__lh_sum', 'mr_y_smri__vol__dsk__cac__lh_sum',
    'mr_y_smri__vol__dsk__cmfrt__lh_sum',  'mr_y_smri__vol__dsk__cn__lh_sum',
    'mr_y_smri__vol__dsk__er__lh_sum',     'mr_y_smri__vol__dsk__ff__lh_sum',
    'mr_y_smri__vol__dsk__ic__lh_sum',     'mr_y_smri__vol__dsk__ins__lh_sum',
    'mr_y_smri__vol__dsk__iprt__lh_sum',   'mr_y_smri__vol__dsk__itmp__lh_sum',
    'mr_y_smri__vol__dsk__lg__lh_sum',     'mr_y_smri__vol__dsk__lobfrt__lh_sum',
    'mr_y_smri__vol__dsk__locc__lh_sum',   'mr_y_smri__vol__dsk__mobfrt__lh_sum',
    'mr_y_smri__vol__dsk__mtmp__lh_sum',   'mr_y_smri__vol__dsk__pactr__lh_sum',
    'mr_y_smri__vol__dsk__pcc__lh_sum',    'mr_y_smri__vol__dsk__pcg__lh_sum',
    'mr_y_smri__vol__dsk__pfrt__lh_sum',   'mr_y_smri__vol__dsk__ph__lh_sum',
    'mr_y_smri__vol__dsk__pob__lh_sum',    'mr_y_smri__vol__dsk__poctr__lh_sum',
    'mr_y_smri__vol__dsk__pop__lh_sum',    'mr_y_smri__vol__dsk__prcn__lh_sum',
    'mr_y_smri__vol__dsk__prctr__lh_sum',  'mr_y_smri__vol__dsk__ptg__lh_sum',
    'mr_y_smri__vol__dsk__ptmp__lh_sum',   'mr_y_smri__vol__dsk__rac__lh_sum',
    'mr_y_smri__vol__dsk__rmfrt__lh_sum',  'mr_y_smri__vol__dsk__sfrt__lh_sum',
    'mr_y_smri__vol__dsk__sm__lh_sum',     'mr_y_smri__vol__dsk__sprt__lh_sum',
    'mr_y_smri__vol__dsk__stmp__lh_sum',   'mr_y_smri__vol__dsk__ttmp__lh_sum',
    # right hemisphere — 34 regions
    'mr_y_smri__vol__dsk__bstmps__rh_sum', 'mr_y_smri__vol__dsk__cac__rh_sum',
    'mr_y_smri__vol__dsk__cmfrt__rh_sum',  'mr_y_smri__vol__dsk__cn__rh_sum',
    'mr_y_smri__vol__dsk__er__rh_sum',     'mr_y_smri__vol__dsk__ff__rh_sum',
    'mr_y_smri__vol__dsk__ic__rh_sum',     'mr_y_smri__vol__dsk__ins__rh_sum',
    'mr_y_smri__vol__dsk__iprt__rh_sum',   'mr_y_smri__vol__dsk__itmp__rh_sum',
    'mr_y_smri__vol__dsk__lg__rh_sum',     'mr_y_smri__vol__dsk__lobfrt__rh_sum',
    'mr_y_smri__vol__dsk__locc__rh_sum',   'mr_y_smri__vol__dsk__mobfrt__rh_sum',
    'mr_y_smri__vol__dsk__mtmp__rh_sum',   'mr_y_smri__vol__dsk__pactr__rh_sum',
    'mr_y_smri__vol__dsk__pcc__rh_sum',    'mr_y_smri__vol__dsk__pcg__rh_sum',
    'mr_y_smri__vol__dsk__pfrt__rh_sum',   'mr_y_smri__vol__dsk__ph__rh_sum',
    'mr_y_smri__vol__dsk__pob__rh_sum',    'mr_y_smri__vol__dsk__poctr__rh_sum',
    'mr_y_smri__vol__dsk__pop__rh_sum',    'mr_y_smri__vol__dsk__prcn__rh_sum',
    'mr_y_smri__vol__dsk__prctr__rh_sum',  'mr_y_smri__vol__dsk__ptg__rh_sum',
    'mr_y_smri__vol__dsk__ptmp__rh_sum',   'mr_y_smri__vol__dsk__rac__rh_sum',
    'mr_y_smri__vol__dsk__rmfrt__rh_sum',  'mr_y_smri__vol__dsk__sfrt__rh_sum',
    'mr_y_smri__vol__dsk__sm__rh_sum',     'mr_y_smri__vol__dsk__sprt__rh_sum',
    'mr_y_smri__vol__dsk__stmp__rh_sum',   'mr_y_smri__vol__dsk__ttmp__rh_sum',
    # bilateral totals
    'mr_y_smri__vol__dsk__lh_sum', 'mr_y_smri__vol__dsk__rh_sum', 'mr_y_smri__vol__dsk_sum',
]

volume = pd.read_parquet(PHENOTYPE_DIR / "mr_y_smri__vol__dsk.parquet")
volume["ID-wave"] = volume["participant_id"].astype(str) + "_" + volume["session_id"].astype(str)
volume = volume.set_index("ID-wave")
volume = volume[CORTICAL_COLS]

print(f"Cortical volumes loaded: {volume.shape[0]:,} scans × {volume.shape[1]} regions")


Cortical volumes loaded: 30,276 scans × 71 regions


## Step 2 — Load subcortical volumes (ASEG)

In [4]:
SUBCORTICAL_COLS = [
    'mr_y_smri__vol__aseg__ab__lh_sum', 'mr_y_smri__vol__aseg__ab__rh_sum',  # amygdala
    'mr_y_smri__vol__aseg__ag__lh_sum', 'mr_y_smri__vol__aseg__ag__rh_sum',  # accumbens
    'mr_y_smri__vol__aseg__cd__lh_sum', 'mr_y_smri__vol__aseg__cd__rh_sum',  # caudate
    'mr_y_smri__vol__aseg__hc__lh_sum', 'mr_y_smri__vol__aseg__hc__rh_sum',  # hippocampus
    'mr_y_smri__vol__aseg__pl__lh_sum', 'mr_y_smri__vol__aseg__pl__rh_sum',  # pallidum
    'mr_y_smri__vol__aseg__pt__lh_sum', 'mr_y_smri__vol__aseg__pt__rh_sum',  # putamen
    'mr_y_smri__vol__aseg__th__lh_sum', 'mr_y_smri__vol__aseg__th__rh_sum',  # thalamus
]

sub_volume = pd.read_parquet(PHENOTYPE_DIR / "mr_y_smri__vol__aseg.parquet")
sub_volume["ID-wave"] = sub_volume["participant_id"].astype(str) + "_" + sub_volume["session_id"].astype(str)
sub_volume = sub_volume.set_index("ID-wave")
sub_volume = sub_volume[SUBCORTICAL_COLS]

print(f"Subcortical volumes loaded: {sub_volume.shape[0]:,} scans × {sub_volume.shape[1]} regions")


Subcortical volumes loaded: 30,276 scans × 14 regions


## Step 3 — Merge cortical + subcortical

In [5]:
T1 = pd.concat([sub_volume, volume], axis=1, join="inner")
print(f"Merged: {T1.shape[0]:,} scans × {T1.shape[1]} regions")

# Scans per wave
wave_counts = T1.index.str.slice(start=-7).value_counts().reindex(
    ["ses-00A", "ses-02A", "ses-04A", "ses-06A"], fill_value=0
)
print("\nScans per wave (before QC):")
print(wave_counts.to_string())


Merged: 30,276 scans × 85 regions

Scans per wave (before QC):
ID-wave
ses-00A    11755
ses-02A     8092
ses-04A     6343
ses-06A     4086


## Step 4 — Apply T1 QC inclusion filter

Keeps only scans where `mr_y_qc__incl__smri__t1_indicator == 1`.

In [8]:
qc = pd.read_parquet(PHENOTYPE_DIR / "mr_y_qc__incl.parquet")
qc["ID-wave"] = qc["participant_id"].astype(str) + "_" + qc["session_id"].astype(str)
qc = qc.set_index("ID-wave")
qc = qc[["mr_y_qc__incl__smri__t1_indicator"]]

# Keep only QC-passing scans
qc_pass = qc[qc["mr_y_qc__incl__smri__t1_indicator"].astype(str) == "1"].index
T1 = T1.loc[T1.index.intersection(qc_pass)]

# Drop any remaining rows with missing values
T1 = T1.dropna()

wave_counts_qc = T1.index.str.slice(start=-7).value_counts().reindex(
    ["ses-00A", "ses-02A", "ses-04A", "ses-06A"], fill_value=0
)
print(f"After QC: {T1.shape[0]:,} scans")
print("\nScans per wave (after QC):")
print(wave_counts_qc.to_string())


After QC: 29,594 scans

Scans per wave (after QC):
ID-wave
ses-00A    11324
ses-02A     7897
ses-04A     6296
ses-06A     4077


## Step 5 — Save

In [9]:
T1.to_csv(OUTPUT_FILE)
print(f"Saved: {OUTPUT_FILE}")
print(f"Shape: {T1.shape[0]:,} scans × {T1.shape[1]} brain regions")


Saved: /Users/nioushad/Documents/Doc_p/myJupyter/normative modeling/code/component/release6/hpc/finalpaper/output/T1_processed.csv
Shape: 29,594 scans × 85 brain regions
